# 第25章 不确定度、解释性与科学可信度

这个 notebook 用一个教学版光度红移数据集演示：bootstrap 区间能告诉我们什么，又不能告诉我们什么；permutation importance 如何做最小特征重要性分析；局部解释更像模型行为说明，而不是物理因果证明；为什么样本一旦跑出训练分布，窄区间也可能不可靠。

## 学习目标

- 区分预测值、误差条和可信度边界
- 理解 bootstrap 覆盖的是哪类波动
- 做最小 permutation importance 和局部解释
- 看到分布外样本如何破坏“看起来稳定”的预测
- 建立解释性不等于因果性的意识

In [ ]:
from __future__ import annotations

import csv
import math
import platform
import random
from pathlib import Path

DATA_PATH = Path('../../data/small/photoz_trust_demo.csv').resolve()

with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = []
    for raw in csv.DictReader(handle):
        rows.append({
            'galaxy_id': raw['galaxy_id'],
            'u_g': float(raw['u_g']),
            'g_r': float(raw['g_r']),
            'r_i': float(raw['r_i']),
            'i_z': float(raw['i_z']),
            'specz': float(raw['specz']),
            'role': raw['role'],
        })

print(f'Loaded {len(rows)} photo-z samples from {DATA_PATH.name}')
role_counts = {}
for row in rows:
    role_counts[row['role']] = role_counts.get(row['role'], 0) + 1
print('role counts:', role_counts)
print('first row:', rows[0])
print('Python version:', platform.python_version())


In [ ]:
feature_names = ['u_g', 'g_r', 'r_i', 'i_z']
train_rows = [row for row in rows if row['role'] == 'train']
test_rows = [row for row in rows if row['role'] == 'test']
edge_rows = [row for row in rows if row['role'] == 'edge']

print('train/test/edge sizes:', len(train_rows), len(test_rows), len(edge_rows))
print('edge ids:', [row['galaxy_id'] for row in edge_rows])


## 一个最小线性光度红移模型

这里继续使用一个可解释的线性 baseline，让后面的不确定度与解释分析都有清楚参照。

In [ ]:
def solve_linear_system(matrix, vector):
    size = len(vector)
    augmented = [row[:] + [value] for row, value in zip(matrix, vector)]
    for col in range(size):
        pivot_row = max(range(col, size), key=lambda row_index: abs(augmented[row_index][col]))
        augmented[col], augmented[pivot_row] = augmented[pivot_row], augmented[col]
        pivot_value = augmented[col][col]
        if abs(pivot_value) < 1e-12:
            raise ValueError('Singular matrix')
        for j in range(col, size + 1):
            augmented[col][j] /= pivot_value
        for row_index in range(size):
            if row_index == col:
                continue
            factor = augmented[row_index][col]
            for j in range(col, size + 1):
                augmented[row_index][j] -= factor * augmented[col][j]
    return [augmented[row_index][size] for row_index in range(size)]

def design_vector(row):
    return [1.0] + [row[name] for name in feature_names]

def fit_ridge_linear_regression(data_rows, ridge=1e-6):
    design = [design_vector(row) for row in data_rows]
    targets = [row['specz'] for row in data_rows]
    parameter_count = len(design[0])
    matrix = []
    vector = []
    for i in range(parameter_count):
        current_row = []
        for j in range(parameter_count):
            value = sum(design_row[i] * design_row[j] for design_row in design)
            if i == j and i != 0:
                value += ridge
            current_row.append(value)
        matrix.append(current_row)
        vector.append(sum(design_row[i] * target for design_row, target in zip(design, targets)))
    return solve_linear_system(matrix, vector)

def predict(coefficients, row):
    return sum(coefficient * value for coefficient, value in zip(coefficients, design_vector(row)))

def rmse(data_rows, coefficients):
    squared_errors = [(predict(coefficients, row) - row['specz']) ** 2 for row in data_rows]
    return math.sqrt(sum(squared_errors) / len(squared_errors))

coefficients = fit_ridge_linear_regression(train_rows)
print('coefficients:', [round(value, 4) for value in coefficients])
print('train RMSE:', round(rmse(train_rows, coefficients), 4))
print('test RMSE:', round(rmse(test_rows, coefficients), 4))
print('edge RMSE:', round(rmse(edge_rows, coefficients), 4))


In [ ]:
evaluation_rows = test_rows + edge_rows
print('Predictions on test and edge samples:')
for row in evaluation_rows:
    prediction = predict(coefficients, row)
    error = prediction - row['specz']
    print(
        row['galaxy_id'],
        row['role'],
        'true={:.3f}'.format(row['specz']),
        'pred={:.3f}'.format(prediction),
        'error={:.3f}'.format(error),
    )


## Bootstrap 区间

这一部分回答的是：如果训练集稍微变化，预测会波动多大。

In [ ]:
def percentile(sorted_values, fraction):
    index = int(fraction * len(sorted_values))
    index = max(0, min(index, len(sorted_values) - 1))
    return sorted_values[index]

random.seed(0)
bootstrap_predictions = {row['galaxy_id']: [] for row in evaluation_rows}
for _ in range(200):
    sampled_train = [random.choice(train_rows) for _ in train_rows]
    sampled_coefficients = fit_ridge_linear_regression(sampled_train)
    for row in evaluation_rows:
        bootstrap_predictions[row['galaxy_id']].append(predict(sampled_coefficients, row))

print('Bootstrap intervals:')
for row in evaluation_rows:
    values = sorted(bootstrap_predictions[row['galaxy_id']])
    lower = percentile(values, 0.16)
    upper = percentile(values, 0.84)
    covered = lower <= row['specz'] <= upper
    print(
        row['galaxy_id'],
        row['role'],
        'interval=({:.3f}, {:.3f})'.format(lower, upper),
        'width={:.3f}'.format(upper - lower),
        'contains_truth={}'.format(covered),
    )


## 特征重要性：permutation importance 的最小版本

这里我们衡量的是模型依赖，不是物理因果。

In [ ]:
baseline_test_rmse = rmse(test_rows, coefficients)
print('Permutation importance on test set:')
for feature_name in feature_names:
    shuffled_values = [row[feature_name] for row in test_rows]
    shuffled_values = shuffled_values[1:] + shuffled_values[:1]
    modified_rows = []
    for row, value in zip(test_rows, shuffled_values):
        modified = dict(row)
        modified[feature_name] = value
        modified_rows.append(modified)
    shuffled_rmse = rmse(modified_rows, coefficients)
    print(feature_name, round(shuffled_rmse - baseline_test_rmse, 4))


## 局部解释：一个 SHAP-like 直觉示例

下面这个做法不是严格 SHAP，只是帮助我们理解“把某个特征替换到基线值后，预测变化多少”。

In [ ]:
training_feature_means = {name: sum(row[name] for row in train_rows) / len(train_rows) for name in feature_names}
target_row = next(row for row in evaluation_rows if row['galaxy_id'] == 'E001')
base_prediction = predict(coefficients, target_row)
print('Local explanation intuition for E001:')
print('  base prediction:', round(base_prediction, 3))
for feature_name in feature_names:
    modified = dict(target_row)
    modified[feature_name] = training_feature_means[feature_name]
    modified_prediction = predict(coefficients, modified)
    delta = base_prediction - modified_prediction
    print(f'  {feature_name}: delta={delta:.3f}')
print('  Note: these deltas describe model behavior, not direct physical causation.')


## 覆盖范围：到训练支持的距离

这一部分回答的是：样本是否还在模型见过的特征空间里。

In [ ]:
feature_means = {}
feature_stds = {}
for feature_name in feature_names:
    values = [row[feature_name] for row in train_rows]
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    feature_means[feature_name] = mean_value
    feature_stds[feature_name] = math.sqrt(variance) or 1.0

def standardized_vector(row):
    return [
        (row[feature_name] - feature_means[feature_name]) / feature_stds[feature_name]
        for feature_name in feature_names
    ]

train_vectors = [standardized_vector(row) for row in train_rows]
print('Distance to training support:')
for row in evaluation_rows:
    vector = standardized_vector(row)
    nearest_distance = min(
        math.sqrt(sum((left - right) ** 2 for left, right in zip(vector, train_vector)))
        for train_vector in train_vectors
    )
    print(row['galaxy_id'], row['role'], round(nearest_distance, 3))
print('Observation: edge samples are far outside the training support even when bootstrap intervals are not extremely wide.')


## 三层可信度视角

把结果重新整理成三个问题：误差怎么样、模型依赖什么、样本是否分布外。

In [ ]:
print('Trust checklist by sample:')
for row in evaluation_rows:
    values = sorted(bootstrap_predictions[row['galaxy_id']])
    lower = percentile(values, 0.16)
    upper = percentile(values, 0.84)
    prediction = predict(coefficients, row)
    support_distance = min(
        math.sqrt(sum((left - right) ** 2 for left, right in zip(standardized_vector(row), train_vector)))
        for train_vector in train_vectors
    )
    print({
        'galaxy_id': row['galaxy_id'],
        'role': row['role'],
        'prediction': round(prediction, 3),
        'truth': round(row['specz'], 3),
        'interval_width': round(upper - lower, 3),
        'support_distance': round(support_distance, 3),
    })


In [ ]:
try:
    from sklearn.inspection import permutation_importance
    from sklearn.linear_model import Ridge
except ModuleNotFoundError:
    print('scikit-learn 未安装；已跳过标准库可信度示例。')
else:
    train_matrix = [[row[name] for name in feature_names] for row in train_rows]
    test_matrix = [[row[name] for name in feature_names] for row in test_rows]
    train_targets = [row['specz'] for row in train_rows]
    model = Ridge(alpha=1e-6)
    model.fit(train_matrix, train_targets)
    importance = permutation_importance(model, test_matrix, [row['specz'] for row in test_rows], n_repeats=20, random_state=0)
    print('sklearn permutation importances:')
    for feature_name, value in sorted(zip(feature_names, importance.importances_mean), key=lambda item: item[1], reverse=True):
        print(feature_name, round(value, 4))


## Trust Statement Export

这一段导出一份最小 Trust Statement 草稿，把预测结果、不确定度、解释性、分布覆盖和失效边界放在同一个可审查文本里。

In [ ]:
RESULTS_DIR = Path('results/part3_evidence_artifacts')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def interval_for(row):
    values = sorted(bootstrap_predictions[row['galaxy_id']])
    lower = percentile(values, 0.16)
    upper = percentile(values, 0.84)
    return lower, upper

def support_distance_for(row):
    vector = standardized_vector(row)
    return min(
        math.sqrt(sum((left - right) ** 2 for left, right in zip(vector, train_vector)))
        for train_vector in train_vectors
    )

sample_lines = []
for row in evaluation_rows:
    lower, upper = interval_for(row)
    prediction = predict(coefficients, row)
    support_distance = support_distance_for(row)
    status = 'out-of-distribution' if row['role'] == 'edge' or support_distance > 2.0 else 'in-distribution teaching range'
    sample_lines.append(
        f"- {row['galaxy_id']} ({row['role']}): pred={prediction:.3f}, truth={row['specz']:.3f}, "
        f"interval=({lower:.3f}, {upper:.3f}), support_distance={support_distance:.3f}, status={status}"
    )

importance_rows = []
for feature_name in feature_names:
    shuffled_values = [row[feature_name] for row in test_rows]
    shuffled_values = shuffled_values[1:] + shuffled_values[:1]
    modified_rows = []
    for row, value in zip(test_rows, shuffled_values):
        modified = dict(row)
        modified[feature_name] = value
        modified_rows.append(modified)
    shuffled_rmse = rmse(modified_rows, coefficients)
    importance_rows.append((feature_name, shuffled_rmse - baseline_test_rmse))
importance_rows.sort(key=lambda item: item[1], reverse=True)
importance_lines = [f"- {name}: delta_rmse={value:.4f}" for name, value in importance_rows]
main_feature = importance_rows[0][0]

artifact = f'''# Ch25 Trust Statement Draft

## Model Output

- Model: ridge-stabilized linear photo-z baseline.
- Main diagnostic: test RMSE={rmse(test_rows, coefficients):.4f}; edge RMSE={rmse(edge_rows, coefficients):.4f}.
- Evaluation samples:
{chr(10).join(sample_lines)}

## Distribution Status

- In-distribution evidence: ordinary test samples have smaller support distances to the training set.
- Out-of-distribution warning: edge samples are farther from training support and have much worse RMSE.

## Uncertainty

- Main uncertainty source estimated here: training-sample resampling variability via bootstrap.
- Boundary: bootstrap intervals do not include all systematic error, selection effects, or out-of-distribution risk.

## Interpretability

- Main feature dependence by permutation check: {main_feature}.
- Permutation summary:
{chr(10).join(importance_lines)}
- Why this is not causal proof: feature dependence describes this fitted model under this dataset and preprocessing, not a physical causal mechanism.

## Failure Boundary

- Known failure region: samples far from training support, especially edge-role objects.
- Human review needed: any object with large support distance, large residual, or scientific use beyond the teaching sample.

## Claim Boundary

- Supported claim: the notebook demonstrates how uncertainty, feature dependence, and training-support checks jointly affect model trust.
- Unsupported claim: the model is not a production photo-z estimator and should not be used for survey science.
'''

output_path = RESULTS_DIR / 'ch25_trust_statement_draft.md'
output_path.write_text(artifact, encoding='utf-8')
print('wrote', output_path)
print('main feature dependence:', main_feature)


## 小结

这个例子说明：可信度不是一个分数，而是一套联合检查。你既要看预测区间，也要看模型依赖什么，还要看样本是不是已经跑出了训练支持范围。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_rows': len(rows),
    'test_rmse': round(rmse(test_rows, coefficients), 4),
    'edge_rmse': round(rmse(edge_rows, coefficients), 4),
    'baseline_test_rmse': round(baseline_test_rmse, 4),
    'python_version': platform.python_version(),
}

print('Trust snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')
